<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network 徽标"  />
    </a>
</p>


<h1>实验：使用 KNN 进行图像分类 </h1>


预计所需时间：**60** 分钟


## 概述


你将使用 K 近邻（k-NN）算法——一种监督式机器学习方法——来对图像进行分类。与其他在训练过程中学习参数的模型不同，k-NN 是一种非参数、基于实例的算法。它会保存带标签的训练数据，并通过在给定输入中寻找最相似的训练实例来进行预测，相似度基于选定的距离度量（例如欧几里得距离）。


## 目标


基于带标签的图像数据，使用 OpenCV 训练并评估一个 k 近邻（k-NN）分类器，以对猫和狗的图像进行分类。


## 目录


本笔记本按以下章节组织：
        <ul>
            <li>[安装并导入库](#Install-and-Import-Libraries)</li>
            <li>[下载图片与标注](#Download-Your-Images-and-Annotations)</li>
            <li>[加载并显示图片](#Load-and-Plot-the-Image)</li>
            <li>[图像处理](#Image-Processing)</li>
            <ul>
                <li>将图片转换为灰度</li>
                <li>调整图片大小</li>
                <li>展平图片</li>
            </ul>
    <li>[训练 k-NN 分类器进行图像分类](#Train-a-k-NN-Classifier-for-Image-Classification) </li>
            <li>[练习](#Practice-Exercise)</li>
        </ul>
    </li>
    



----


<h2 id="Install-and-Import-Libraries">安装并导入库</h2>


In [ ]:
!pip install opencv-python-headless --upgrade --quiet
!pip install numpy pandas matplotlib seaborn imutils scikit-learn

**导入库**


用于数据处理和可视化的库：


In [ ]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
from imutils import paths
import seaborn as sns
import random
import time
from datetime import datetime
import requests
import zipfile
import json
import random

用于图像预处理和分类的库：


In [ ]:
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

用于操作系统的库：


In [ ]:
import io
import os

<h2 id="Download-Your-Images-and-Annotations">下载图片与标注</h2>


现在，让我们初始化并从 URL 下载图片。


In [ ]:
# ZIP 文件的 URL
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/RdukW75jUsonAnS20t3n_g/training-an-image-classifier-w-2025-05-22-t-10-27-47-719-z.zip"

# 向该 URL 发送 GET 请求
response = requests.get(url)

# 检查请求是否成功
if response.status_code == 200:
    # 从下载内容中打开 zip 文件
    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        zip_ref.extractall("cats_dogs")  # 提取到目标文件夹
    print("Download and extraction complete.")
else:
    print("Failed to download file:", response.status_code)


👈🏾 你可以检查侧面板，查看图片是否已下载。


在图像分类中，标注是描述图像内容的标签或元数据。这些标签对于训练监督式机器学习模型至关重要。标注将保存在一个 JSON 文件中，其中图像名称作为键，`dog` 或 `cat` 作为标签对象。

让我们查看刚下载的标注格式。以下代码将只显示前 5 个标注。


In [ ]:
# 定义标注 JSON 文件的路径
annotations_path = "cats_dogs/training-an-image-classifier-w-2025-05-22-t-10-27-47-719-z/_annotations.json"

# 加载 JSON 文件
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# 现在安全地访问前 5 个条目
first_five = {k: annotations["annotations"][k] for k in list(annotations["annotations"])[:5]}
first_five


<h1 id="Load-and-Plot-the-Image">加载并显示图片</h1>
我们将使用 <code>OpenCV</code> 库和 k-NN 分类器来训练并对你的图像进行分类。开始之前，让我们先获取图片并查看其中一些。


我们将随机挑选图片并查看：使用 `cv2.imread` 和 `matplotlib` 库读取并显示一张随机图片。


<h2>加载标注文件</h2>


In [ ]:
# 定义基础文件夹路径
base_folder = "cats_dogs/training-an-image-classifier-w-2025-05-22-t-10-27-47-719-z"

# 标注 JSON 文件的路径
annotations_path = os.path.join(base_folder, "_annotations.json")

# 加载 JSON 数据
with open(annotations_path, "r") as f:
    annotations = json.load(f)

print("Annotations loaded successfully!")

<h2>随机选择一张图片</h2>


In [ ]:
# 从标注中随机选择一张图片
random_image_name = random.choice(list(annotations["annotations"].keys()))

# 获取该图片的标签
label = annotations["annotations"][random_image_name][0]["label"]

print(f"Random image selected: {random_image_name}")
print(f"Label: {label}")

<h2>读取并转换图片</h2>


In [ ]:
# 构建图片的完整路径
image_path = os.path.join(base_folder, random_image_name)

# 使用 OpenCV 读取图片
img = cv2.imread(image_path)
if img is None:
    raise FileNotFoundError(f"Image not found at {image_path}")
print("Full image path:", image_path)


<h2>将 BGR 转换为 RGB 并显示</h2>


我们将图片从 `BGR` 颜色空间转换为 `RGB`，以便使用 matplotlib 正确显示。这是因为 OpenCV 默认以 BGR 格式读取图片，这一约定最初因早期相机制造商和软件供应商的广泛使用而被采用。


In [ ]:
# 将图片颜色从 BGR 转换为 RGB
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 使用 matplotlib 绘图
plt.figure(figsize=(6, 6))
plt.imshow(img_rgb)
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

In [ ]:
# 如果你绘制 img（BGR 图片），你会发现颜色空间的差异
plt.imshow(img)
plt.axis('off')
plt.title(f"Label: {label}")
plt.show()

<h1 id="Image-Processing">图像处理</h1>


这里我们将讨论以下内容：
 <ul>
            <li>将图片转换为灰度</li>
            <li>调整图片大小</li>
            <li>展平图片 </li>
            </ul>
    </li>


我们将先用单张图片开始处理，然后再对所有图片重复相同的过程。

要在数据集上执行 KNN，我们需要先处理数据。我将使用示例图片（img_rgb）来解释每一行代码。


### 将图片转换为灰度 


将图片转换为灰度——灰度可以简化算法并降低计算需求。


In [ ]:
sample_image = cv2.cvtColor(img_rgb,cv2.COLOR_BGR2GRAY)
plt.figure(figsize=(10,10))
plt.imshow(sample_image, cmap = "gray")
plt.show()

### 调整图片大小 

调整图片大小有助于算法更快训练。


In [ ]:
sample_image = cv2.resize(img_rgb, (32, 32))
plt.imshow(sample_image, cmap = "gray")
plt.show()

### 展平图片

将图片转换为 numpy 数组，便于算法处理和识别。


In [ ]:
pixels = sample_image.flatten()
pixels

### 对所有图片重复上述过程


现在我们将重复上述过程，加载并处理你标注过的所有图片，并为每张图片设置标签。KNN 是监督式机器学习算法，因此我们必须为机器显式地创建标签。

根据数据量大小，这可能需要运行一段时间……


加载标注文件：已经完成，跳过即可


准备图片路径和标签列表


In [ ]:
# 获取数据集文件夹中所有图片文件的路径
image_paths = list(paths.list_images(base_folder))

# 创建空列表以存储图片数据和对应的标签
train_images = []
train_labels = []

# 从标注中提取类别标签列表（例如 ['dog', 'cat']）
class_object = annotations['labels']


处理并为每张图片标注标签


In [ ]:
from tqdm import tqdm
# 使用进度条处理每张图片
for image_path in tqdm(image_paths, desc="Loading images"):
    filename = os.path.basename(image_path)

    # 如果不在标注中则跳过
    if filename not in annotations["annotations"]:
        continue  # 默默地跳过

    # 加载图片
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    image = cv2.resize(image, (32, 32))
    pixels = image.flatten()

    # 获取标签
    tmp_label = annotations["annotations"][filename][0]['label']
    label = class_object.index(tmp_label)

    # 追加
    train_images.append(pixels)
    train_labels.append(label)


创建 <code>train_images</code> 和 <code>train_labels</code> 数组。<code>OpenCV</code> 要求训练样本为 <code>float32</code> 类型数组，训练标签为形状 <code>(标签数量, 1)</code> 的数组。我们可以通过在训练样本的 numpy 数组上指定 <code>astype('float32')</code>，将训练标签转换为整数，并使用 <code>reshape</code> 将其变为 <code>(标签数量, 1)</code> 来实现。当你打印 <code>train_labels</code> 时，数组看起来会是这样：<code>[[1], [0], ..., [0]]</code></p>


In [ ]:
train_images = np.array(train_images).astype('float32')
train_labels = np.array(train_labels)

重塑并查看训练标签


In [ ]:
train_labels = train_labels.astype(int)
train_labels = train_labels.reshape((train_labels.size,1))
print("First 5 labels:\n", train_labels[:5])

<h1 id="Train-a-k-NN-Classifier-for-Image-Classification">训练 k-NN 分类器进行图像分类</h1>


检查图片数量及其标签：


In [ ]:
print(f"Number of images: {len(train_images)}")
print(f"Number of labels: {len(train_labels)}")


按你选择的测试比例将数据划分为训练集和测试集：


In [ ]:
test_size = 0.2
train_samples, test_samples, train_labels, test_labels = train_test_split(
    train_images, train_labels, test_size=test_size, stratify=train_labels,random_state=0)
print(f"Number of train_samples: {len(train_samples)}")
print(f"Number of test_samples: {len(test_samples)}")

为了训练 KNN 模型，我们将使用 <code>OpenCV</code> 库中的 <code>cv2.ml.KNearest_create()</code>。我们需要定义用于分类的近邻数量，这就是超参数 k。该参数 k 可以在训练或模型验证过程中进行调整或调优。拟合训练图片和测试图片，并获得模型的准确率得分。

我们将尝试多个 <code>k</code> 值，以找到适合当前数据集的最佳值。<code>k</code> 指的是在多数投票过程中包含的近邻数量。

<i>注意：</i>根据数据集大小，运行可能需要几秒钟。


In [ ]:
import cv2
import sys

print("Python executable:", sys.executable)
print("OpenCV version:", cv2.__version__)
print("Has cv2.ml?", hasattr(cv2, "ml"))
print("cv2 path:", cv2.__file__)

In [ ]:
# 记录开始时间以测量训练耗时
start_datetime = datetime.now()

# 使用 OpenCV 的机器学习模块创建 KNN 模型
knn = cv2.ml.KNearest_create()

# 使用训练样本和对应标签训练模型
# cv2.ml.ROW_SAMPLE 表示 train_samples 中的每一行都是一个独立样本
knn.train(train_samples, cv2.ml.ROW_SAMPLE, train_labels)

# 定义要评估的不同 K 值
k_values = [1, 2, 3, 4, 5]
k_result = []  # 存储每个 K 值的预测结果

# 遍历每个 K 值，在测试样本上测试模型
for k in k_values:
    ret, result, neighbours, dist = knn.findNearest(test_samples, k=k)
    k_result.append(result)  # 保存该 K 值的结果

# 展平结果数组，方便后续比较
flattened = []
for res in k_result:
    # 每个 `res` 都是二维数组，将其展平为一维列表
    flat_result = [item for sublist in res for item in sublist]
    flattened.append(flat_result)

# 记录结束时间并打印训练与预测共耗时多久
end_datetime = datetime.now()
print('Training Duration: ' + str(end_datetime - start_datetime))


我们将计算每个 <code>k</code> 值对应的准确率，即有多少百分比的图片被正确分类？我们还将创建混淆矩阵，以便更全面地评估分类模型。


In [ ]:
# 创建空列表以存储每个 K 的准确率和混淆矩阵
accuracy_res = []
con_matrix = []

# 遍历每个 K 值的结果
for k_res in k_result:
    # 定义类别标签（例如 0 = 猫，1 = 狗）
    label_names = [0, 1]

    # 计算预测结果与真实标签的混淆矩阵
    cmx = confusion_matrix(test_labels, k_res, labels=label_names)
    con_matrix.append(cmx)

    # 检查哪些预测与真实标签匹配
    matches = k_res == test_labels

    # 统计预测正确的数量
    correct = np.count_nonzero(matches)

    # 计算准确率（百分比）
    accuracy = correct * 100.0 / result.size
    accuracy_res.append(accuracy)

# 将每个 K 值的准确率存入字典（键 = K，值 = 准确率）
res_accuracy = {k_values[i]: accuracy_res[i] for i in range(len(k_values))}

# 按 K 值排序结果，便于阅读或绘图
list_res = sorted(res_accuracy.items())


准确率


In [ ]:
print("\nAccuracy per k:")
for k, acc in list_res:
    print(f"k = {k}: {acc:.2f}%")

**混淆矩阵快速指南**


混淆矩阵是分类问题的性能度量工具。它是一张结合预测值和实际值的表格。y 轴为 `真实` 标签，x 轴为 `预测` 标签。本示例将关注二分类器，即“是或否”模型。

<table>
  <tr>
    <td>&nbsp;</td>
    <td>预测：否</td>
    <td>预测：是</td>
  </tr>
  <tr>
    <td>真实：否</td>
    <td>30</td>
    <td>30</td>
  </tr>
  <tr>
    <td>真实：是</td>
    <td>10</td>
    <td>50</td>
  </tr>
</table>

在这个矩阵中，我们可以看到有两个类别。例如，如果我们要预测一张图片是否是热狗，“是”表示热狗，“否”表示不是热狗。我们共有 120 个预测结果，其中分类器预测“是”80 次、“否”40 次，但实际上有 60 个“是”和 60 个“否”。

谈到混淆矩阵时，我们会涉及以下几个术语：
* 真正例（TP）：模型预测为“是”，实际也是“是”
* 真负例（TN）：模型预测为“否”，实际也是“否”
* 假正例（FP）：模型预测为“是”，但实际是“否”
* 假负例（FN）：模型预测为“否”，但实际是“是”

让我们结合本示例具体来看：

<table>
  <tr>
    <td>&nbsp;</td>
    <td>预测：否</td>
    <td>预测：是</td>
  </tr>
  <tr>
    <td>真实：否</td>
    <td>TN = 30</td>
    <td>FP = 30</td>
    <td>60</td>
  </tr>
  <tr>
    <td>真实：是</td>
    <td>FN = 10</td>
    <td>TP = 50</td>
    <td>60</td>
  </tr>
  <tr>
    <td>&nbsp;</td>
    <td>40</td>
    <td>80</td>
  </tr>
</table>

**准确率** 是模型预测正确的数量除以预测总数，即 (TP+TN)/预测总数。


现在让我们可视化混淆矩阵：


In [ ]:
t = 0  # 初始化 k 值计数器

# 遍历每个 k 值对应的混淆矩阵
for array in con_matrix:
    # 将混淆矩阵数组转换为 pandas DataFrame 以便更好地可视化
    df_cm = pd.DataFrame(array)
    
    # 设置字体大小以提高可读性
    sns.set(font_scale=1.4)
    
    # 根据 DataFrame 创建热力图
    sns.heatmap(df_cm, annot=True, annot_kws={"size": 16}, fmt=".0f")  # fmt=".0f" 确保显示整数
    
    # 更新 k 计数器
    t += 1
    
    # 为每个混淆矩阵图设置标题
    title = "Confusion Matrix for k equals " + str(t)
    plt.title(title)
    
    # 显示图像
    plt.show()


我们将绘制准确率图，查看哪个 k 值最高，即有多少百分比的图片被正确分类？


In [ ]:
## 绘制准确率变化图
x, y = zip(*list_res)
plt.plot(x, y)
plt.show()

我们将找到最佳的 <code>k</code> 值来训练模型，并在我们的图片上测试模型：


In [ ]:
k_best = max(list_res,key=lambda item:item[1])[0]
k_best

保存训练好的 kNN 模型


In [ ]:
knn.save('knn_samples.yml')

<h2 id="Practice-Exercise">练习</h2>
### 使用上传的图片测试模型


上传你的图片，看看它是否能被正确分类。
<p><b>如何上传图片的说明：</b></p>
使用上传按钮，从你的本地机器上传一张图片：
<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction.png" width="300"  />
</center>


上传后的图片将位于你当前工作的目录中。要在新单元格中读取图片，请使用 <code>cv2.imread</code> 并传入图片名称。例如，我将 <code>anothercar.jpg</code> 上传到当前工作目录后，可以使用 <code>cv2.imread("anothercar.jpg")</code> 读取。

<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-CV0101EN-SkillsNetwork/images/instruction2.png" width="300"  />
</center>


或者使用下方图片进行测试。


In [ ]:
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/A3xVgxTJVdrZTtBwGVbEIw/Cat.jpg"
!wget "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/KZnFQiZj3e_sQKIyfvHvTA/dog.jpg"

加载模型


In [ ]:
knn = cv2.ml.KNearest_create()
knn = knn.load('knn_samples.yml')


将下方的 `Replace your_uploaded_file` 替换为你目录中看到的图片名称。如果你使用的是笔记本提供的下载图片，则使用 `Cat.jpg` 或 `dog.jpg`。


In [ ]:
my_image = cv2.imread("test2.png")
## 看看图片长什么样
image = cv2.cvtColor(my_image, cv2.COLOR_BGR2RGB)
plt.imshow(image)
plt.axis('off')
plt.show()

将图片转换为灰度——灰度可以简化算法并降低计算需求。


In [ ]:
my_image = cv2.cvtColor(my_image,cv2.COLOR_BGR2GRAY)

调整图片大小以减小尺寸：


In [ ]:
my_image = cv2.resize(my_image, (32, 32))
image = cv2.cvtColor(my_image, cv2.COLOR_BGR2RGB)
plt.imshow(image)
plt.axis('off')
plt.show()

将图片展平为 numpy 数组：


In [ ]:
pixel_image = my_image.flatten()
pixel_image = np.array([pixel_image]).astype('float32')

对图片进行分类并打印模型结果：


In [ ]:
ret,result,neighbours,dist = knn.findNearest(pixel_image,k=k_best)
print("Nearest Neighbours' Labels:\n", neighbours)
predicted_index = int(ret)
predicted_label = annotations['labels'][predicted_index]
print(f"Your image was classified as a **{predicted_label}**")


当我们打印出近邻时，它会告诉你 k 个最近的类别，并使用多数投票机制来决定你的图片应被分为哪一类。


## 恭喜！

你已经成功完成了一个使用 k 近邻（k-NN）算法和 OpenCV 的完整图像分类流程！


<h2>作者</h2>


[Aije Egwaikhide](https://www.linkedin.com/in/aije-egwaikhide/)

[Sathya Priya](https://www.linkedin.com/in/sathya-priya-06120a17a/) 


<!--<table>
    <tr>
        <th>日期（YYYY-MM-DD）</th>
        <th>版本</th>
        <th>修改者</th>
        <th>变更说明</th>
    </tr>
        <tr>
        <td>2025-06-25</td>
        <td>1.3</td>
        <td>Sathya</td>
        <td>创建并将实验转换为 JupyterCurrent 笔记本</td>
    </tr>
    </tr>
        <tr>
        <td>2021-05-25</td>
        <td>1.2</td>
        <td>Kathy</td>
        <td>修改多个区域</td>
    </tr>
    <tr>
        <td>2021-05-25</td>
        <td>1.2</td>
        <td>Yasmine</td>
        <td>修改多个区域</td>
    </tr>
    <tr>
        <td>2021-04-10</td>
        <td>1.1</td>
        <td>Aije</td>
        <td>修正语法错误</td>
    </tr>
     <tr>
        <td>2021-04-09</td>
        <td>1.0</td>
        <td>Aije</td>
        <td>更新为新模板</td>
    </tr>
    <tr>
        <td>2021-02-24</td>
        <td>0.1</td>
        <td>Aije</td>
        <td>创建实验的原始版本</td>
    </tr>
</table>-->


<h3 align="center"> &#169; IBM Corporation。保留所有权利。 <h3/>
